In [4]:
from utils import add_project_root_to_sys_path
add_project_root_to_sys_path()
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

import polars as pl
import matplotlib.pyplot as plt
import mplhep as hep
plt.style.use(hep.style.ATLAS)
import numpy as np

In [5]:
def get_rename_function(prefix: str):
    def rename_function(col_name: str) -> str:
        if col_name ==  'id':
            return col_name

        return f'{prefix}.{col_name}'
    return rename_function

In [6]:
qat_inference = pl.scan_parquet('/home/lucasbanunes/data/neuralnet_jobs/sbrt_new/qat_mlp_v1/threshold_fit_inference/inference_results.parquet').rename(get_rename_function('qat'))
mlp_inference = pl.scan_parquet('/home/lucasbanunes/data/neuralnet_jobs/sbrt_new/mlp_v0/threshold_fit_inference/inference_results.parquet').rename(get_rename_function('mlp'))
fold_df = pl.scan_parquet('/home/lucasbanunes/data/ringer-datasets/datasets/mc21_isabela_qt_2sigma_restriction/standard_binning_kfold.parquet').rename(get_rename_function('fold'))
analysis_df = qat_inference.join(mlp_inference, on=['id'], how='inner').join(fold_df, on=['id'], how='inner')
analysis_df.collect_schema()

Schema([('id', UInt64),
        ('qat.loose.output', Float32),
        ('qat.loose.prediction', Boolean),
        ('qat.medium.output', Float32),
        ('qat.medium.prediction', Boolean),
        ('qat.tight.output', Float32),
        ('qat.tight.prediction', Boolean),
        ('qat.very_loose.output', Float32),
        ('qat.very_loose.prediction', Boolean),
        ('mlp.loose.output', Float32),
        ('mlp.loose.prediction', Boolean),
        ('mlp.medium.output', Float32),
        ('mlp.medium.prediction', Boolean),
        ('mlp.tight.output', Float32),
        ('mlp.tight.prediction', Boolean),
        ('mlp.very_loose.output', Float32),
        ('mlp.very_loose.prediction', Boolean),
        ('fold.label', Boolean),
        ('fold.kfold', UInt8)])

In [ ]:
analysis_df = analysis_df.select(
    'label', 'qat.tight.output', 'mlp.tight.output'
)
hist_columns = {
    'qat.tight.output': {
        'label': 'FP',
        'marker': 'o',
        'linestyle': '-',
        'df': analysis_df,
    },
    'mlp.tight.output.prediction': {
        'label': 'QAT',
        'df': analysis_df,
        'marker': '^',
        'linestyle': '--'
    },
    'ptq.prediction': {
        'label': 'PTQ',
        'df': analysis_df,
        'col': 'ptq.prediction',
        'marker': 's',
        'linestyle': '-.'
    }
}

In [ ]:
fig = plt.figure(layout="tight")
grid_spec = GridSpec(8, 1, figure=fig)
hist_ax = fig.add_subplot(grid_spec[:4, 0])
# fig, ax = plt.subplots(3, 1)
hist_ax.grid(linestyle='--', alpha=0.1, color='k')
colors = [
    'red',
    'blue',
    'green',
    'orange',
    'purple',
    'cyan',
    'magenta',
]
class_colors = [
    'C0',
    'C1'
]
for i, (class_label, hist_col) in enumerate(product(range(2), hist_columns)):
    hist_config = hist_columns[hist_col]
    color = class_colors[class_label]
    n, bins = np.histogram(
        hist_config['df'].filter(pl.col('label') == class_label)[hist_col].to_numpy(),
        bins=50,
        density=True
    )
    # n, bins, _ =  ax.hist(
    #     hist_config['df'].filter(pl.col('label') == class_label)[hist_col].to_numpy(),
    #     bins=50,
    #     alpha=0.5,
    #     linestyle=hist_config['linestyle'],
    #     color='white',
    #     label=None,
    #     histtype='step',
    #     linewidth=2,
    #     density=True
    # )
    x = (bins[:-1] + bins[1:])/2
    y = n

    hist_ax.scatter(
        x, y,
        color=color,
        s=20,
        alpha=0.7,
        edgecolor='k',
        linewidth=0.5,
        marker=hist_config['marker'],
        label=f"{hist_config['label']} - Classe {class_label}"
    )
    hist_columns[hist_col][f'y_class_{class_label}'] = y
    hist_columns[hist_col][f'x_class_{class_label}'] = x
# hist_ax.set_xlabel('Saída do Modelo', fontsize='small')
hist_ax.set_ylabel('Densidade', fontsize='medium')
hist_ax.set_yscale('log')
hist_ax.set_title('Distribuição da saída dos modelos')
hist_ax.legend(
    fontsize='small',
    loc='upper center',
    ncols=2
)
hist_ax.tick_params(
    axis='x',          # Apply to x and y axes
    which='both',        # Target major ticks
    # bottom=False,
    labelbottom=False
)
hist_ax.tick_params(
    axis='y',          # Apply to x and y axes
    which='both',        # Target major ticks
    # bottom=False,
    labelsize='small'
)

classes_ratios = {
    'PTQ / FP': {
        'ax': fig.add_subplot(grid_spec[4:6, 0]),
        'numerator': 'ptq.prediction',
        'denominator': 'fp.prediction',
        'tick_params':[
            {'axis': 'x', 'which': 'both', 'labelbottom': False},
            {'axis': 'y', 'which': 'both', 'labelsize': 'small'}
        ]
    },
    'QAT / FP': {
        'ax': fig.add_subplot(grid_spec[6:8, 0]),
        'numerator': 'qat.prediction',
        'denominator': 'fp.prediction',
        'tick_params':[
            {'axis': 'x', 'which': 'both', 'labelsize': 'small'},
            {'axis': 'y', 'which': 'both', 'labelsize': 'small'}
        ]
    }
}

for ratio_label, ratio_config in classes_ratios.items():
    ratio_ax = ratio_config['ax']
    ratio_ax.grid(linestyle='--', alpha=0.1, color='k')
    ratio_ax.set_ylabel(ratio_label, fontsize='medium')
    ratio_ax.axhline(1, color='k', linestyle='--', alpha=0.5)
    for param in ratio_config['tick_params']:
        ratio_ax.tick_params(**param)
    for color, class_label in zip (class_colors, range(2)):
        
        numerator_config = hist_columns[ratio_config['numerator']]
        denominator_config = hist_columns[ratio_config['denominator']]
        ptq_fp_ratio = numerator_config[f'y_class_{class_label}'] / denominator_config[f'y_class_{class_label}']
        ratio_ax.scatter(
            numerator_config[f'x_class_{class_label}'], ptq_fp_ratio,
            color=color,
            s=20,
            alpha=0.7,
            edgecolor='k',
            linewidth=0.5,
            marker=numerator_config['marker'],
            label=f'Classe {class_label}'
        )

    # ratio_ax.legend(
    #     fontsize='small',
    #     # loc='upper center',
    #     ncols=3
    # )


fig.tight_layout()